# 02 — Panel Scoring (Pre / Post Windows)

Scores each panel user's text in the **pre-baseline** (August) and **post-outcome** (December–May)
windows using the trained SVM classifiers, then merges with exposure labels to build the analysis panel.

**Inputs:**
- `data/processed_v2/posts_clean.jsonl` + `comments_clean.jsonl` (from notebook 01)
- `data/processed_v2/exposure_labels_v2.parquet` (from notebook 07)
- `models/clf_anxiety.joblib`, `clf_depression.joblib`, `clf_stress.joblib`

**Output:** `data/processed_v2/panel_scores_v2.parquet`

| Column | Description |
|--------|-------------|
| author | Reddit username |
| cycle | 1 or 2 |
| exposed | bool — whether user commented on an anchor thread |
| pre_mh_score | Mean SVM MH score across Aug posts/comments |
| pre_n_posts | Count of posts/comments in Aug |
| post_mh_score | Mean SVM MH score across Dec–May posts/comments |
| post_n_posts | Count of posts/comments in Dec–May |

**Cycle windows:**

| | Cycle 1 | Cycle 2 |
|-|---------|----------|
| Pre baseline | Aug 1–31, 2023 | Aug 1–31, 2024 |
| Post outcome | Dec 1, 2023 – May 31, 2024 | Dec 1, 2024 – May 31, 2025 |

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from datetime import datetime, timezone
from pathlib import Path

ROOT      = Path('..').resolve()
DATA_V2   = ROOT / 'data' / 'processed_v2'
MODEL_DIR = ROOT / 'models'

POSTS_CLEAN    = DATA_V2 / 'posts_clean.jsonl'
COMMENTS_CLEAN = DATA_V2 / 'comments_clean.jsonl'
EXPOSURE_PATH  = DATA_V2 / 'exposure_labels_v2.parquet'
OUT_PATH       = DATA_V2 / 'panel_scores_v2.parquet'

# Cycle windows (inclusive on both ends)
CYCLES = {
    1: {
        'pre_start':  datetime(2023,  8,  1, tzinfo=timezone.utc),
        'pre_end':    datetime(2023,  8, 31, 23, 59, 59, tzinfo=timezone.utc),
        'post_start': datetime(2023, 12,  1, tzinfo=timezone.utc),
        'post_end':   datetime(2024,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
    2: {
        'pre_start':  datetime(2024,  8,  1, tzinfo=timezone.utc),
        'pre_end':    datetime(2024,  8, 31, 23, 59, 59, tzinfo=timezone.utc),
        'post_start': datetime(2024, 12,  1, tzinfo=timezone.utc),
        'post_end':   datetime(2025,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
}

print('Paths OK:', all(p.exists() for p in [POSTS_CLEAN, COMMENTS_CLEAN, EXPOSURE_PATH]))

## 1) Load panel users

In [ ]:
exposure = pd.read_parquet(EXPOSURE_PATH)
print(f'Panel users: {exposure["author"].nunique():,}  |  rows: {len(exposure):,}')
print(exposure['exposed'].value_counts())
panel_users = set(exposure['author'])

## 2) Load clean corpus and assign cycle windows

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

posts    = load_jsonl(POSTS_CLEAN)
comments = load_jsonl(COMMENTS_CLEAN)
print(f'Posts: {len(posts):,}  |  Comments: {len(comments):,}')

In [ ]:
# Combine into a single list; keep only panel users and relevant windows
def parse_dt(s):
    dt = datetime.fromisoformat(s)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt


def assign_window(dt):
    """Return (cycle, window) tuple or None if outside all windows."""
    for cycle, w in CYCLES.items():
        if w['pre_start'] <= dt <= w['pre_end']:
            return cycle, 'pre'
        if w['post_start'] <= dt <= w['post_end']:
            return cycle, 'post'
    return None


records = []
for r in posts + comments:
    author = r.get('author')
    if author not in panel_users:
        continue
    try:
        dt = parse_dt(r['created_dt'])
    except Exception:
        continue
    result = assign_window(dt)
    if result is None:
        continue
    cycle, window = result
    records.append({'author': author, 'cycle': cycle, 'window': window,
                    'clean_text': r.get('clean_text', '')})

corpus = pd.DataFrame(records)
print(f'Panel records in scoring windows: {len(corpus):,}')
print(corpus.groupby(['cycle', 'window']).size())

## 3) Load SVM classifiers

In [ ]:
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def score_texts(texts):
    """Return mean_mh_score array for a list of strings."""
    anx = sigmoid(clf_anx.decision_function(texts))
    dep = sigmoid(clf_dep.decision_function(texts))
    str_ = sigmoid(clf_str.decision_function(texts))
    return np.stack([anx, dep, str_], axis=1).mean(axis=1)

## 4) Score all records

In [ ]:
texts = corpus['clean_text'].tolist()
print(f'Scoring {len(texts):,} records...')
corpus['mean_mh_score'] = score_texts(texts)
print('Done.')
corpus['mean_mh_score'].describe().round(4)

## 5) Aggregate per (author, cycle, window)

In [ ]:
agg = (
    corpus
    .groupby(['author', 'cycle', 'window'])
    .agg(mh_score=('mean_mh_score', 'mean'), n_posts=('mean_mh_score', 'count'))
    .reset_index()
)

# Pivot window → columns
pre  = agg[agg['window'] == 'pre' ].rename(columns={'mh_score': 'pre_mh_score',  'n_posts': 'pre_n_posts' }).drop(columns='window')
post = agg[agg['window'] == 'post'].rename(columns={'mh_score': 'post_mh_score', 'n_posts': 'post_n_posts'}).drop(columns='window')

scores = pre.merge(post, on=['author', 'cycle'], how='inner')
print(f'Users with both pre and post observations: {len(scores):,}')

## 6) Merge with exposure labels

In [ ]:
panel = exposure.merge(scores, on=['author', 'cycle'], how='inner')
print(f'Final panel rows: {len(panel):,}')
print(f'Unique users:     {panel["author"].nunique():,}')
print(f'\nCoverage: {100 * panel["author"].nunique() / len(panel_users):.1f}% of panel users have pre+post scores')
print('\nExposure breakdown:')
print(panel.groupby(['cycle', 'exposed']).size())

In [ ]:
# Score distribution check
print('Pre-period MH scores:')
print(panel.groupby('exposed')['pre_mh_score'].describe().round(4))
print('\nPost-period MH scores:')
print(panel.groupby('exposed')['post_mh_score'].describe().round(4))

## 7) Save

> **Approval gate:** Review coverage stats and score distributions above before running this cell.

In [ ]:
out_cols = ['author', 'cycle', 'exposed', 'pre_mh_score', 'pre_n_posts', 'post_mh_score', 'post_n_posts']
panel[out_cols].to_parquet(OUT_PATH, index=False)
print(f'Saved {len(panel):,} rows → {OUT_PATH}')
panel[out_cols].head()